In [ ]:
import pandas as pd
import textwrap
from urllib.parse import urlparse, parse_qs

INPUT_CSV = "data/songs.csv"
OUTPUT_QMD = "music/songs.qmd"

def extract_youtube_id(url):
    """
    Extract YouTube video ID from common URL formats.
    """
    parsed = urlparse(url)

    if "youtube.com" in parsed.netloc:
        return parse_qs(parsed.query).get("v", [None])[0]
    elif "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/")
    return None

df = pd.read_csv(INPUT_CSV)
df = df.sort_index(ascending=False)

print_lyric = True
print_mood_scores = True

with open(OUTPUT_QMD, "w", encoding="utf-8") as f:
    # Quarto header
    f.write("---\n")
    f.write("title: \"Song Rankings\"\n")
    f.write("format: html\n")
    f.write("toc: false\n")
    f.write("---\n\n")

    f.write("The greatest songs of all time according to yours truly.\n\n")

    for i in df.index:
        row = df.loc[i]
        ranking = i + 1
        artist = row["Artist"]
        title = row["Title"]
        album = row["Album"]
        year = row["Year"]
        url = row["URL"]
        genre = row["Genre"]
        lyric = row["Lyric"]

        video_id = extract_youtube_id(url)

        f.write(f"## {ranking}. {artist} - {title}\n")
        f.write(f"**Album:** *{album}* ({year})\\\n")
        f.write(f"**Genres:** {genre}\n\n")

        if pd.notna(lyric) and print_lyric:
            f.write(f"*{lyric.replace(' / ', '\\\n')}*\n\n")

        if print_mood_scores:
            for col in df.columns:
                if col.startswith("Mood_"):
                    mood = col.split("_")[1]
                    score = int(df.loc[i, col])  # 0–5
                    max_score = 5

                    percent = score / 5 * 100
                    # score_string = textwrap.dedent(f"""
                    #     <div style="display:grid; grid-template-columns: 64px 160px; align-items:center;">
                    #     <div>{mood}</div>
                    #     <div class="progress" style="height:14px;">
                    #         <div class="progress-bar bg-info"
                    #             role="progressbar"
                    #             style="width:{percent}%;">
                    #         </div>
                    #     </div>
                    #     </div>
                    #     """)
                    score_string = f"""<div style="display:grid; grid-template-columns: 64px 160px; align-items:center; margin-bottom:0px;"><div>{mood}</div><div class="progress" style="height:14px;"><div class="progress-bar bg-primary" role="progressbar" style="width:{percent}%;"></div></div></div>"""
                    #score_string = "●" * score + "○" * (max_score - score)
                    f.write(f"{score_string}")
            f.write("<br>")


        # Video
        f.write("<details><summary>Listen here</summary>")
        # f.write(f"{{{{< video {url} >}}}}\n\n")
        if video_id:
            f.write(f'<div class="lite-youtube-style"><lite-youtube videoid="{video_id}"></lite-youtube></div>\n')
        f.write("</details>")
        f.write("\n\n")
        #f.write("\n---\n\n")

print(f"Generated {OUTPUT_QMD}")


Generated music/songs.qmd


In [10]:
df

,Artist,Title,Album,Year,Genre,URL,MV
5,Pink Floyd,Wish You Were Here,Wish You Were Here,1975,Art Rock,https://www.youtube.com/watch?v=K6qj09OHvjw,True
4,Mac Miller,Good News,Circles,2020,Neo-Soul,https://www.youtube.com/watch?v=aIHF7u9Wwiw,True
3,The Devin Townsend Band,Deadhead,Accelerated Evolution,2003,Progressive Metal,https://www.youtube.com/watch?v=tJz-L3SbrLE,False
2,Fela Anikulapo Kuti and Afrika 70,Zombie,Zombie,1977,Afrobeat,https://www.youtube.com/watch?v=Qj5x6pbJMyU,False
1,FKA Twigs,Cellophane,Magdalene,2019,Art Pop,https://www.youtube.com/watch?v=YkLjqFpBh84,True
0,Sam Cooke,A Change Is Gonna Come,Ain’t That Good News,1964,Soul,https://www.youtube.com/watch?v=1oZ6Z9xRxo8,False
